# Data Preprocessing for Single-Cell Analysis

This notebook demonstrates the essential preprocessing steps required before running cell segmentation and analysis:

## Key Preprocessing Steps
1. **Dataset Organization**: Standardize file naming and structure
2. **Train-Test Split**: Divide data into training and testing sets for model development
3. **Z-stack Processing**: Combine 2D slices to create 3D volumes for analysis


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [4]:
## Setup and Imports

from pathlib import Path
import json
import tempfile
import numpy as np
import tifffile as tiff

# Import preprocessing modules
from src.preprocessing import train_test_split_directory
from src.utils.conversion import combine_2d_to_3d
# from src.preprocessing.dataset_split import split_dataset_dict

# File handling utilities
from src.utils.file_utils import (
    BF_IF_FileHandler               # Specialized for brightfield and IF images
)

## 1. Dataset Organization and Train-Test Split

This section demonstrates how to:

1. **Standardize file naming conventions** - Creates a consistent naming pattern for all images
2. **Split the dataset into training and test sets** - Essential for unbiased model evaluation
3. **Preserve group structure** - Ensures related images (same field of view) stay in the same set

In [5]:
# Define input and output directory paths

# Input data directory - contains original microscopy files 
data_path = "../data/Plate 2426"

# Output directory for 2D preprocessed data
output_dir = "../data/Plate 2426_preprocessed_2D_split_out"

# Output directory for 3D preprocessed data
output_3d_dir = "../data/Plate 2426_preprocessed_3D_split_out"

# Create output directory if it doesn't exist
Path(output_dir).mkdir(parents=True, exist_ok=True)
Path(output_3d_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
test_split = 0.4
random_seed = 42

# Split the dataset into training and test sets
split_json = train_test_split_directory(
    data_dir = data_path,
    output_dir = output_dir,
    test_size = test_split,
    random_state = random_seed,
    file_handler= BF_IF_FileHandler(),
)

print("Dataset split into train and test sets!")
print(f"Train set: {len(split_json['train_files']['image'])} images")
print(f"Test set: {len(split_json['test_files']['image'])} images")

### Inspect a saved split manifest

In [6]:
# Load any previously saved manifest and summarise it.
# USER: replace with your actual path, or set OUTPUT_DIR in the cell above first.
MANIFEST_PATH = Path(output_dir) / "dataset_split.json"

if not MANIFEST_PATH.exists():
    print(f"No manifest found at {MANIFEST_PATH}. Run the split cell above first.")
else:
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)

    print("Split summary")
    print("─" * 40)
    for split, file_types in manifest.items():
        for ftype, paths in file_types.items():
            print(f"  {split:15s}  {ftype:10s}  {len(paths):>4d} files")

Split summary
────────────────────────────────────────
  train_files      mask        2875 files
  train_files      BF          1440 files
  train_files      image       1440 files
  test_files       mask        1919 files
  test_files       BF           960 files
  test_files       image        960 files


## 2. Z-Stack to 3D Volume Conversion

This section demonstrates how to combine 2D z-stack images into 3D volumes for analysis.

The process involves:
1. Identifying related z-slices using filename patterns
2. Sorting slices by z-position
3. Combining slices into a single 3D TIFF file

The pattern parameter uses regular expressions to identify z-stack components in filenames.

2D microscopy acquisitions produce one TIFF per optical section (z-slice). This step stacks related slices into a single 3D volume, which is required as input for blur analysis and cell segmentation.

Files are grouped by matching a regular-expression **pattern** against each filename. The default pattern captures:
- **base name** — everything before the first `_z`
- **z-index** — the integer after `_z`
- **channel suffix** — optional `_BF` or `_Cells` (or similar) before the extension

In [7]:
# Combine 2D to 3D
combine_2d_to_3d(
    input_dir=output_dir,
    output_dir=output_3d_dir,
    pattern=r"(.+?)_z(\d+)(?:_(BF|Cells|w2))?\.(tif)",     # Regex pattern to identify z-stacks : will fail without this fixed pattern
    recursive=True,
)


Finding 2D files: 100%|██████████| 7200/7200 [00:00<00:00, 484004.69it/s]


Found 240 groups of 2D images to combine into 3D volumes.
Example groups:
  p2426_I16_t1_w2: 30 files
  p2426_I06_t1_Cells: 30 files
  p2426_C03_t1_Cells: 30 files
  p2426_B08_t1_Cells: 30 files
  p2426_B14_t1_Cells: 30 files


Combining to 3D: 100%|██████████| 240/240 [03:14<00:00,  1.24it/s]

Successfully combined 240 2D image sets into 3D volumes in ../data/Plate 2426_preprocessed_3D_split_out


---
## 3. Synthetic sample demo

### 3.1 Synthetic data with two channels, two wells

In [7]:
with tempfile.TemporaryDirectory() as tmp:
    input_dir  = Path(tmp) / "slices"
    output_dir = Path(tmp) / "volumes"
    input_dir.mkdir()

    rng_local = np.random.default_rng(0)
    wells    = ["A01", "B02"]
    channels = ["BF", "Cells"]

    # Write 2D z-slices: p2126_A01_t1_z1_BF.tif, ..., p2126_B02_t1_z3_Cells.tif
    for well in wells:
        for ch in channels:
            for z in range(1, 4):  # z1, z2, z3
                img = rng_local.integers(0, 4000, (64, 64), dtype=np.uint16)
                tiff.imwrite(str(input_dir / f"p2126_{well}_t1_z{z}_{ch}.tif"), img)

    combine_2d_to_3d(
        input_dir=str(input_dir),
        output_dir=str(output_dir),
        pattern=r"(.+?)_z(\d+)_(BF|Cells)\.(tif)",
    )

    volumes = sorted(output_dir.glob("*.tif"))
    print(f"\nCreated {len(volumes)} 3D volumes:")
    for v in volumes:
        vol = tiff.imread(str(v))
        print(f"  {v.name}  →  shape {vol.shape}  dtype {vol.dtype}")

Finding 2D files: 100%|██████████| 12/12 [00:00<00:00, 33711.75it/s]


Found 4 groups of 2D images to combine into 3D volumes.
Example groups:
  p2126_A01_t1_BF: 3 files
  p2126_B02_t1_Cells: 3 files
  p2126_B02_t1_BF: 3 files
  p2126_A01_t1_Cells: 3 files


Combining to 3D: 100%|██████████| 4/4 [00:00<00:00, 86.95it/s]

Successfully combined 4 2D image sets into 3D volumes in /var/folders/xn/vsqnhgdd4zdbrg1_47j52jk00000gn/T/tmpzhmecgge/volumes

Created 4 3D volumes:
  p2126_A01_t1_BF_3d.tif  →  shape (3, 64, 64)  dtype uint16
  p2126_A01_t1_Cells_3d.tif  →  shape (3, 64, 64)  dtype uint16
  p2126_B02_t1_BF_3d.tif  →  shape (3, 64, 64)  dtype uint16
  p2126_B02_t1_Cells_3d.tif  →  shape (3, 64, 64)  dtype uint16


### 3.2 Split a directory (demo)

`train_test_split_directory` discovers all files automatically, splits them, copies them into `train/` and `test/` subdirectories, and saves a `dataset_split.json` manifest.

In [8]:
with tempfile.TemporaryDirectory() as tmp:
    data_dir   = Path(tmp) / "Plate 2126"
    output_dir = Path(tmp) / "split_output"
    data_dir.mkdir()

    # Three wells, two z-slices each, plus matching masks.
    for well_row, well_col in [("A", 1), ("B", 2), ("C", 3)]:
        for z in (1, 2):
            (data_dir / f"t1_{well_row}{well_col:02d}_s1_w1_z{z}.tif").write_bytes(b"img")
        for z in (0, 1):
            (data_dir / f"Cells_R{well_col}-C{well_col}-F0-Z{z}-T0.tif").write_bytes(b"msk")

    result = train_test_split_directory(
        data_dir=str(data_dir),
        output_dir=str(output_dir),
        test_size=0.34,
        random_state=42,
        file_handler=BF_IF_FileHandler(),
    )

    print("── Split result ──────────────────────────────────────")
    print(f"  Train images : {len(result['train_files']['image'])}")
    print(f"  Test  images : {len(result['test_files']['image'])}")
    print(f"  Train masks  : {len(result['train_files']['mask'])}")
    print(f"  Test  masks  : {len(result['test_files']['mask'])}")

    manifest_path = output_dir / "dataset_split.json"
    print(f"\n  Manifest written to : {manifest_path.relative_to(tmp)}")

    # Inspect the manifest.
    with open(manifest_path) as f:
        manifest = json.load(f)

    print("\n── dataset_split.json (sample) ───────────────────────")
    for split in ("train_files", "test_files"):
        for ftype, paths in manifest[split].items():
            print(f"  {split}.{ftype}: {[Path(p).name for p in paths[:2]]}{'...' if len(paths) > 2 else ''}")

No files found for pattern 'FilePattern(pattern='p(\\d+)_t(\\d+)_([A-Z])(\\d+)_z(\\d+)_(.+)', groups=['plate', 'time', 'row', 'col', 'z', 'type'], output_format='{plate}_{row}{col}_z{z}_w{type}.tif')' in /var/folders/xn/vsqnhgdd4zdbrg1_47j52jk00000gn/T/tmpp0ubeytv/Plate 2126 for type 'processed'


── Split result ──────────────────────────────────────
  Train images : 4
  Test  images : 2
  Train masks  : 4
  Test  masks  : 2

  Manifest written to : split_output/dataset_split.json

── dataset_split.json (sample) ───────────────────────
  train_files.mask: ['p2126_A01_t1_z1_Cells.tif', 'p2126_A01_t1_z2_Cells.tif']...
  train_files.image: ['p2126_A01_t1_z1_w1.tif', 'p2126_A01_t1_z2_w1.tif']...
  test_files.mask: ['p2126_C03_t1_z1_Cells.tif', 'p2126_C03_t1_z2_Cells.tif']
  test_files.image: ['p2126_C03_t1_z1_w1.tif', 'p2126_C03_t1_z2_w1.tif']
